In [1]:
import cobra

import pandas as pd

from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Alphabet import generic_dna

import multiprocessing
from multiprocessing import Pool
# from tqdm import tqdm

import scipy.stats as st
from bs4 import BeautifulSoup
import urllib
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings

import requests, sys, json, re
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *


load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [ ]:
# MANE SELECTED transcripts and protein sequences
ids = pd.read_csv(local_data_path + 'raw/MANE.GRCh38.v0.9.summary.txt', sep = '\t')

psim_me = ids.loc[:, ['Ensembl_Gene', 'HGNC_ID', '#NCBI_GeneID', 'Ensembl_nuc', 'Ensembl_prot', 'symbol', 'name', 'chr_strand']]
psim_me.columns = ['ENSG_ID', 'HGNC_ID', 'NCBI_ID', 'ENST_ID', 'ENSP_ID', 'GENE_SYMBOL', 'GENE_NAME', 'CHR_STRAND']

polyA = pd.read_csv(local_data_path + 'processed/polyA_length.csv', index_col = 0)
psim_me['POLYA_LENGTH'] = psim_me.GENE_SYMBOL.map(dict(zip(polyA.index.tolist(), polyA.MEAN.tolist())))

sequence mapping

In [ ]:
# add protein sequences
protein = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_protein.faa', "fasta"))
p_map = dict()
for p in protein:
    p_map[p.id] = str(p.seq)
psim_me['PROTEIN_SEQ'] = psim_me.ENSP_ID.map(p_map)

# add mrna sequence
mrna = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_rna.fna', "fasta"))
m_map = dict()
for m in mrna:
    m_map[m.id] = str(m.seq)
psim_me['MRNA_SEQ'] = psim_me.ENST_ID.map(m_map)

premrna more complicated because FTP doesn't have ENSG to full gene sequence

In [ ]:
# # # add premrna sequence - no FTP file for this, parallelize REST API instead
# def get_premrna_seq(ensg_id, counter):
#     print(counter)
#     try:
#         hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
#         return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text # introns and UTRs
#     except:
#         return float('nan')

# # pool = Pool(processes = multiprocessing.cpu_count())
# # premrna = pool.starmap(get_premrna_seq, zip(psim_me['ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# # pool.close()

# # with open(local_data_path + 'interim/premrna_sequences.txt', 'w') as f:
# #     for seq in premrna:
# #         if type(seq) != str:
# #             seq = str(seq)
# #         f.write(seq + '\n')

# premrna = open(local_data_path + 'interim/premrna_sequences.txt', 'r').read().splitlines()
###NOT ALL DOWNLOADED, SO RERUNNING ON THOSE THAT DIDNT DOWNLOAD
# fail = 'You have exceeded the limit of 15 requests per second; please reduce your concurrent connections'
# fail_index = [i for i in range(len(premrna)) if premrna[i] == fail]
# pool = Pool(processes = 4)
# corrected = pool.starmap(get_premrna_seq, zip(psim_me.loc[fail_index, 'ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# pool.close()
# for idx, seq in dict(zip(fail_index, corrected)).items():
#     premrna[idx] = seq

# with open(local_data_path + 'interim/premrna_sequences_v2.txt', 'w') as f:
#     for seq in premrna:
#         if type(seq) != str:
#             seq = str(seq)
#         f.write(seq + '\n')
premrna2 = open(local_data_path + 'interim/premrna_sequences_v2.txt', 'r').read().splitlines()
psim_me['PREMRNA_SEQ'] = premrna2

psim_me.loc[psim_me[psim_me.PREMRNA_SEQ == 'nan'].index, 'PREMRNA_SEQ'] = float('nan')

for i in psim_me.index:
    if type(psim_me.loc[i, 'PREMRNA_SEQ']) == str:
        if len(psim_me.loc[i,'MRNA_SEQ']) > len(psim_me.loc[i,'PREMRNA_SEQ']):
            psim_me.loc[i,'PREMRNA_SEQ'] = float('nan')
            

In [ ]:
# formatting
def transcribe(x):
    try: 
        return str(Seq(x).transcribe())
    except:
        return float('nan')

psim_me['MRNA_SEQ'] = psim_me['MRNA_SEQ'].apply(lambda x: transcribe(x))
psim_me['PREMRNA_SEQ'] = psim_me['PREMRNA_SEQ'].apply(lambda x: transcribe(x))

uniprot_map = pd.read_csv(local_data_path + 'raw/gencode.v34.metadata.SwissProt', sep = '\t', header = None)
psim_me['UNIPROT_ID'] = psim_me['ENST_ID'].map(dict(zip(uniprot_map[0].tolist(), uniprot_map[1].tolist())))

psim_human = pd.read_csv(root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/PSIM_HUMAN.tab', 
                         sep = '\t')

cols = ['SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'Location']
for col in cols:
    psim_me[col] = psim_me['UNIPROT_ID'].map(dict(zip(psim_human.Entry.tolist(), psim_human[col].tolist())))

for i in psim_me.Location.dropna().index:
    a = psim_me.loc[i, 'Location']
    psim_me.loc[i, 'Location'] = a.split('[')[1].split(']')[0]

# Recon 2.2 corrections

In [6]:
# human_model = cobra.io.load_matlab_model(root_path + 'MammalianSecretoryRecon/MODELS/RECON2_2.mat')
# # human_model_2 = cobra.io.load_json_model(local_data_path + 'raw/RECON3D.json')

# # two genes in recon2_2 have HGNC:HGNC:### rather than HGNC:###, the following code corrects that issue

# # since the two genes are involved in the same three reactions, simply need to rewrite these three reactions
# # rather than looping through
# genes_to_duplicate = [gene.id for gene in human_model.genes if gene.id.count(':') > 1]
# g0, g1 = genes_to_duplicate[0], genes_to_duplicate[1]  
# g_i = human_model.genes.get_by_id(g0)
# r_i = list(g_i.reactions)
# g_c = human_model.genes.get_by_id(g0[5:])

# # the following line of code gets rid of both genes in genes to duplicate since they are incolved in the same
# # reations
# human_model.remove_reactions(r_i, remove_orphans=True)
# for r in r_i:
#     r0 = r.gene_reaction_rule.replace(g0, g0[5:])
#     r.gene_reaction_rule = r0.replace(g1, g1[5:])

# # recon2.2 does not have nuclear AMP, adding the transport reaction to model
# amp_n = cobra.Metabolite('amp[n]')
# amp_n.name = 'AMP(2-)'
# amp_n.elements = {'C': 10, 'H': 12, 'N': 5, 'O': 7, 'P': 1}
# amp_n.compartment = 'n'
# amp_n.charge = -2


# amp_transport = cobra.Reaction('AMPtn')
# amp_transport.name = 'AMP nuclear transport'
# amp_c = human_model.metabolites.get_by_id('amp[c]')
# amp_transport.add_metabolites({amp_c: -1, amp_n:1})
# amp_transport.lower_bound = -1000
# human_model.add_reaction(amp_transport)

# # trp_L transport to mitochondria
# trp_m = cobra.Metabolite('trp_L[m]')
# trp_m.name = 'L-tryptophan'
# trp_m.elements = {'C': 11, 'H': 12, 'N': 2, 'O': 2}
# trp_m.compartment = 'm'
# trp_m.charge = 0

# trp_transport = cobra.Reaction('trp_L_MITOCHONDRIAL_MATRIXtn')
# trp_transport.name = 'Mitochondrial matrix transport of L-tryptophan'
# trp_c = human_model.metabolites.get_by_id('trp_L[c]')
# trp_transport.add_metabolites({trp_c: -1, trp_m:1})
# trp_transport.lower_bound = -1000
# human_model.add_reaction(trp_transport)

# aa_ids = ['arg_L[c]', 'asn_L[c]', 'asp_L[c]', 'cys_L[c]', 'glu_L[c]', 'gln_L[c]', 
#          'his_L[c]', 'ile_L[c]', 'leu_L[c]', 'met_L[c]', 'phe_L[c]', 'pro_L[c]', 'thr_L[c]', 'trp_L[c]', 
#          'tyr_L[c]', 'val_L[c]']
# for aa_id in aa_ids:
#     aa_c = human_model.metabolites.get_by_id(aa_id)
#     aa_x = aa_c.copy()
#     aa_x.id = aa_x.id.replace('[c]', '[x]')
#     aa_x.compartment = 'x'

#     aa_transport = cobra.Reaction(aa_x.id.split('[')[0] + '_PEROXISOMALtn')
#     aa_transport.name = 'Peroxisomal transport of ' + aa_x.name
#     aa_transport.add_metabolites({aa_c: -1, aa_x:1})
#     aa_transport.lower_bound = -1000
#     human_model.add_reaction(aa_transport)
# cobra.io.save_json_model(human_model, local_data_path + 'processed/corrected_recon2_2.json')

# # if metabolite not in compartment or transport reaction not in comaprtemtn


human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')

# You are here
must fill out missing machinery sequences

In [ ]:
metabolic_machinery = sorted(set([gene.id for gene in human_model.genes]))

# map sec machinery rxn GPRs from entrez to HGNC IDs
rxnGPR = open(root_path + "MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/rxnGPRs_HUMAN.txt").read().splitlines()
line1 = rxnGPR[0]
rxnGPR = rxnGPR[1:]
secretory_machinery = []
for i in rxnGPR:
    secretory_machinery += re.findall(r'\d+', i)
secretory_machinery = sorted(set(secretory_machinery))
entrez_hgnc_map = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
entrez_hgnc_map = entrez_hgnc_map.loc[entrez_hgnc_map['NCBI gene ID'].dropna().index,:]
entrez_hgnc_map['NCBI gene ID'] = entrez_hgnc_map['NCBI gene ID'].astype('int64').astype(str)
entrez_hgnc_map = entrez_hgnc_map[entrez_hgnc_map['NCBI gene ID'].isin(secretory_machinery)]
secretory_machinery = entrez_hgnc_map['HGNC ID'].tolist()

entrez_hgnc_map = dict(zip(entrez_hgnc_map['NCBI gene ID'].tolist(),entrez_hgnc_map['HGNC ID'].tolist()))

for i in range(len(rxnGPR)):
    mach = re.findall(r'\d+', rxnGPR[i])
    for m in mach:
        rxnGPR[i] = rxnGPR[i].replace(m, entrez_hgnc_map[m])

rxnGPR = [line1] + rxnGPR

with open(local_data_path + 'processed/rxnGPRs_HUMAN_HGNCID.txt', 'w') as f:
    for i in rxnGPR:
        f.write(i + '\n')

a = len(set(metabolic_machinery))
b = len(set(metabolic_machinery).difference(psim_me.loc[:, ['PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'HGNC_ID']].dropna().HGNC_ID.tolist()))

print('{} of {} RECON2.2 machinery mapped'.format(a-b, a))




a = len(set(secretory_machinery))
b = len(set(secretory_machinery).difference(psim_me.loc[:,['PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'HGNC_ID']].dropna().NCBI_ID.apply(lambda x: x.split('GeneID:')[1]).tolist()))

print('{} of {} secretory machinery mapped'.format(a-b, a))

In [ ]:
psim_me.to_csv(local_data_path + 'processed/psim_me.csv')